# Experiment 4: Image Classification using CNN (MNIST Dataset)

## 📌 Objective
Develop, train, and evaluate a **Convolutional Neural Network (CNN)** to classify handwritten digits (0–9) from the **MNIST dataset** (70,000 grayscale images, 28×28 pixels).

### Architecture Overview
- **Conv Block 1**: Conv2D (32 filters, 3×3) → BatchNorm → ReLU → MaxPool2D (2×2)
- **Conv Block 2**: Conv2D (64 filters, 3×3) → BatchNorm → ReLU → MaxPool2D (2×2)
- **Classifier**: Flatten → Dense (128) → ReLU → Dropout (0.5) → Dense (10) → Softmax / CrossEntropy

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Append src directory to module path
sys.path.insert(0, os.path.abspath("../src"))

from data_loader import load_config, prepare_mnist_dataloaders, explore_dataset_summary
from model_builder import build_cnn_model, summarize_model
from training import train_model
from evaluation import evaluate_model_on_test_set, save_evaluation_csv, generate_classification_report
from visualization import (
    plot_training_history,
    plot_confusion_matrix,
    plot_sample_predictions,
    plot_misclassified_samples,
    plot_learned_filters
)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

## 1. Load Configuration & Prepare DataLoaders

In [ ]:
config = load_config("../config/hyperparameters.json")
# Set relative data directory for notebook execution
config["dataset"]["raw_dir"] = "../data/raw/mnist"
config["output"]["model_path"] = "../models/cnn_model.pt"

train_loader, val_loader, test_loader = prepare_mnist_dataloaders(config)
explore_dataset_summary(train_loader, test_loader)

## 2. Visualize Sample Training Digits

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"Digit: {labels[i].item()}", fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Augmented MNIST Training Images", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Build & Summarize CNN Architecture

In [ ]:
model = build_cnn_model(config)
summarize_model(model)

## 4. Train Model with Validation Monitoring & Early Stopping

In [ ]:
history = train_model(model, train_loader, val_loader, config)

## 5. Evaluate Model on Held-out Test Set

In [ ]:
metrics = evaluate_model_on_test_set(model, test_loader, config)
generate_classification_report(metrics, config, "../results/classification_report.txt")

## 6. Visualizations: Curves, Confusion Matrix, Predictions & Filters

In [ ]:
plot_training_history(history, "../results/training_history.png")
plot_confusion_matrix(metrics["confusion_matrix"], "../results/confusion_matrix.png")
plot_sample_predictions(metrics, "../results/sample_predictions.png")
plot_misclassified_samples(metrics, "../results/misclassified_samples.png")
plot_learned_filters(model, "../results/learned_filters.png")